3. Potential Fraud Detection :
Thinking Process : 
Step 1 : User Activity : Grouped transactions by user to calculate their total transaction count, total debit and debit-to-credit ratio.
Step 2 : Calculate Unusual activities : Used Z-score to rank users by how much their activity deviated from the average. Higher the score, higher the fraud possiblity.

In [10]:
import pandas as pd
import numpy as np

In [11]:
def load_and_preprocess_data(file_path):
    try:
        df = pd.read_csv(file_path)
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df.dropna(subset=['value'], inplace=True)
        return df
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return None

In [12]:
def engineer_user_features(df):
    debits = df[df['transaction_type'] == 'DEBIT']
    credits = df[df['transaction_type'] == 'CREDIT']

    user_profiles = df.groupby('adv_id').agg(
        transaction_count=('value', 'count'),
        avg_transaction_value=('value', 'mean')
    )

    debit_summary = debits.groupby('adv_id').agg(
        debit_count=('value', 'count'),
        total_debit_value=('value', 'sum')
    )

    credit_summary = credits.groupby('adv_id').agg(
        credit_count=('value', 'count'),
        total_credit_value=('value', 'sum')
    )

    user_profiles = user_profiles.merge(debit_summary, on='adv_id', how='left')
    user_profiles = user_profiles.merge(credit_summary, on='adv_id', how='left')

    user_profiles.fillna(0, inplace=True)

    user_profiles['debit_ratio'] = user_profiles['debit_count'] / user_profiles['transaction_count']

    return user_profiles

In [13]:
def calculate_anomaly_scores(user_profiles):
    features_for_scoring = [
        'transaction_count',
        'avg_transaction_value',
        'total_debit_value',
        'debit_ratio'
    ]

    for feature in features_for_scoring:
        mean = user_profiles[feature].mean()
        std = user_profiles[feature].std()
        if std > 0:
            user_profiles[f'{feature}_zscore'] = (user_profiles[feature] - mean) / std
        else:
            user_profiles[f'{feature}_zscore'] = 0

    user_profiles['anomaly_score'] = user_profiles[[f'{col}_zscore' for col in features_for_scoring]].abs().sum(axis=1)
    
    return user_profiles.sort_values(by='anomaly_score', ascending=False)

In [14]:
def main():
    file_path = '3_wallet_data.csv'
    
    df = load_and_preprocess_data(file_path)
    
    if df is not None:
        user_profiles = engineer_user_features(df)
        anomalous_users = calculate_anomaly_scores(user_profiles)
        
        print("Top Most Unusual Users by Wallet Activity\n")
        
        print(anomalous_users.head(10)[[
            'transaction_count',
            'total_debit_value',
            'debit_ratio',
            'anomaly_score'
        ]])

if __name__ == "__main__":
    main()

Top Most Unusual Users by Wallet Activity

                                      transaction_count  total_debit_value  \
adv_id                                                                       
1d255507-8308-4c5d-b278-823d56538326                  5          147453.25   
aa14e9a1-70d3-4e87-bb19-3c9cb90a91cc                121         1139664.11   
1dc15540-da1c-45ad-a2c9-381167e47c82                121         1139664.11   
fb261412-6700-4238-8240-ff5c06f5bf3f                121         1139664.11   
6da00477-3c1a-455e-8ddb-aec86ece2e36                121         1139664.11   
2f901805-ef14-48fd-8b53-9b863151df4d                121         1139664.11   
f57e6f5f-4423-4e5e-ab51-e8d2faf50939                121         1139664.11   
83998d57-995f-4e3f-96e6-77dd070b6803                121         1139664.11   
6e7b3f8d-8165-4fd7-8afc-9da3031b13bb                121         1139664.11   
581b565a-0498-4419-95a7-efa15970728a                121         1139664.11   

                    